In [1]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
import sys
sys.path.append('..')
from NES_VMC import NESTotalAnsatz, create_machine, create_machine_matrix, create_single_machine, \
    ha, SingleStateAnsatz, NESFermionHopRule, compute_qgt, nes_vmc_gradient
import optax
from jax.flatten_util import ravel_pytree
import time

print('='*60)
print('NH₃ NES-VMC 计算前三个能级 (K=3)')
print('='*60)

/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: You can use flax.linen, flax.nnx and equinox to define neural networks.

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能：0.0000 eV
E1 = -0.87542794 Ha  |  激发能：3.8107 eV
E2 = -0.42938376 Ha  |  激发能：15.9482 eV
E3 = -0.26922131 Ha  |  激发能：20.3064 eV
NH₃ NES-VMC 计算前三个能级 (K=3)


## 1. NH₃ 分子定义与 FCI 基准

In [2]:
from pyscf import gto, scf, fci

# NH₃ 分子几何结构
# N-H 键长 ~1.008 Å, H-N-H 角 ~106.7°
bond_length = 1.008
angle = 106.7 * jnp.pi / 180

# 三个H原子位于金字塔形底部
geometry = [
    ('N', (0., 0., 0.)),
    ('H', (bond_length * jnp.sin(angle), 0., -bond_length * jnp.cos(angle))),
    ('H', (-bond_length * jnp.sin(angle) * jnp.cos(jnp.pi/3), bond_length * jnp.sin(angle) * jnp.sin(jnp.pi/3), -bond_length * jnp.cos(angle))),
    ('H', (-bond_length * jnp.sin(angle) * jnp.cos(jnp.pi/3), -bond_length * jnp.sin(angle) * jnp.sin(jnp.pi/3), -bond_length * jnp.cos(angle)))
]

# 创建分子对象，使用 STO-3G 基组
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# 计算 FCI 精确基准能量
cisolver = fci.FCI(mf)
cisolver.nroots = 4  # 计算前4个态用于参考
E_fcis, fcivec = cisolver.kernel()

print('='*60)
print('NH₃ FCI 基准能量')
print('='*60)
for i, e in enumerate(E_fcis[:3]):
    exc = (e - E_fcis[0]) * 27.2114
    print(f'E{i} = {e:.8f} Ha  |  激发能：{exc:.4f} eV')

# 分子轨道信息
n_orb = mol.nao_nr()
n_elec = mol.nelectron
print(f'\n轨道数: {n_orb}, 电子数: {n_elec}')

NH₃ FCI 基准能量
E0 = -55.51258121 Ha  |  激发能：0.0000 eV
E1 = -55.06757883 Ha  |  激发能：12.1091 eV
E2 = -55.00874767 Ha  |  激发能：13.7100 eV

轨道数: 8, 电子数: 10


## 2. 希尔伯特空间与哈密顿量设置

In [3]:
# 从 PySCF 分子创建 NetKet 哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

# NH₃: 10电子 (5个自旋向上, 5个自旋向下)
# 在 STO-3G 基组下有 7 个轨道
n_orbitals = mol.nao_nr()
n_alpha = n_elec // 2 + n_elec % 2  # 6
n_beta = n_elec // 2  # 5

print(f'轨道数: {n_orbitals}')
print(f'Alpha电子数: {n_alpha}, Beta电子数: {n_beta}')

# 希尔伯特空间设置
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=n_orbitals,
    s=1/2,
    n_fermions_per_spin=(n_alpha, n_beta)
)
print(f'希尔伯特空间维度: {hi.n_states}')

# NES 扩展副本数 K=3 (计算前3个能级: 基态 + 2个激发态)
K = 2
hi_ext = hi ** K
SINGLE_SIZE = hi.size
#print(f'扩展希尔伯特空间维度: {hi_ext.n_states}')

轨道数: 8
Alpha电子数: 5, Beta电子数: 5
希尔伯特空间维度: 3136


## 3. 神经网络波函数 (Ansatz) 初始化

In [5]:
import jax
import jax.numpy as jnp
import netket as nk

SINGLE_SIZE = hi.size

@nk.utils.struct.dataclass
class NESFermionHopRule(nk.sampler.rules.MetropolisRule):
    edges: jnp.ndarray
    K: int = nk.utils.struct.static_field()
    single_size: int = nk.utils.struct.static_field()

    def _check_duplicate(self, sigma_ext):
        """NES约束：子组态不重复
        🔥 核心修复：返回【标量布尔值】，匹配while_loop初始值形状
        """
        sub = sigma_ext.reshape((-1, self.K, self.single_size))
        # 原代码返回数组 → 改为 .squeeze() 压缩成标量！
        return jnp.any(jnp.all(sub[...,1:,:] == sub[...,0:1,:], axis=-1), axis=-1).squeeze()

    def transition(self, sampler, machine, parameters, state, rng, sigma):
        """跃迁规则（完全不变）"""
        batch_size = sigma.shape[0]
        key1, key2 = jax.random.split(rng)

        e_idx = jax.random.randint(key1, (batch_size,), 0, self.edges.shape[0])
        sel_e = self.edges[e_idx]
        i, j = sel_e[:,0], sel_e[:,1]

        sigma_cand = sigma.at[jnp.arange(batch_size),i].set(sigma[jnp.arange(batch_size),j])
        sigma_cand = sigma_cand.at[jnp.arange(batch_size),j].set(sigma[jnp.arange(batch_size),i])

        invalid = self._check_duplicate(sigma_cand)
        new_sigma = jnp.where(invalid[:, None], sigma, sigma_cand)

        return new_sigma, None

    def random_state(self, sampler, machine, parameters, state, rng):
        """随机态生成（仅修复标量形状）"""
        sigma_shape = state.σ.shape
        hilbert = sampler.hilbert

        def gen_single(key):
            max_tries = 100
            def cond(c): 
                return (c[0] < max_tries) & c[2]
            
            def body(c):
                tries, k, _, _ = c
                k, k_new = jax.random.split(k)
                s = hilbert.random_state(k_new)
                is_dup = self._check_duplicate(s)  # 现在是标量！
                return (tries + 1, k, is_dup, s)
            
            # 初始值 c[2] = True（标量布尔值），和body返回值形状完全匹配
            init_c = (0, key, True, hilbert.random_state(key))
            final_c = jax.lax.while_loop(cond, body, init_c)
            tries, _, is_dup, s = final_c
            return jax.lax.cond(is_dup, lambda: hilbert.random_state(key), lambda: s)
        
        keys = jax.random.split(rng, sigma_shape[0])
        return jax.vmap(gen_single)(keys)

In [6]:
# 创建 NES Total Ansatz
hidden_dim = 32
total_ansatz = NESTotalAnsatz(
    n_spin_orbitals=n_orbitals * 2,  # 考虑自旋
    n_states=K,
    hidden_dim=hidden_dim,
    rngs=nnx.Rngs(42)
)

# 创建总机器和矩阵机器
total_machine, total_graphdef, total_params = create_machine(total_ansatz)
total_matrix_machine, _, _ = create_machine_matrix(total_ansatz)

# 创建单个态的机器列表
single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)

print(f'参数总数: {ravel_pytree(total_params)[0].shape[0]}')

参数总数: 3266


## 4. NES 采样器设置

In [7]:
# 定义费米子跃迁边 (根据轨道生成)
single_edges = []
for i in range(n_orbitals):
    for j in range(i+1, n_orbitals):
        # 每个轨道对 (i, j) 对应两个自旋通道
        single_edges.append((i, j))
        single_edges.append((i + n_orbitals, j + n_orbitals))

single_edges = tuple(single_edges)
print(f'跃迁边数: {len(single_edges)}')

# 构建扩展空间的跃迁边
ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for (i, j) in single_edges:
        ext_edges.append((i + offset, j + offset))
ext_edges = jnp.array(ext_edges)

# 创建 NES 采样规则
nes_rule = NESFermionHopRule(edges=ext_edges, K=K, single_size=SINGLE_SIZE)

# 创建采样器
N_CHAINS = 16
nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=N_CHAINS,
    sweep_size=100
)

print(f'采样器: {N_CHAINS} 链, 每链采样长度 200, 热身 100')

跃迁边数: 56
采样器: 16 链, 每链采样长度 200, 热身 100


## 5. 优化器设置

In [8]:
# 优化器设置
optimizer = optax.sgd(learning_rate=0.002)
opt_state = optimizer.init(total_params)

# 训练超参数
N_WARMUP = 30
N_SAMPLES_PER_CHAIN = 200
N_ITER = 300

print(f'训练参数: 学习率=0.002, 迭代={N_ITER}, K={K}')

训练参数: 学习率=0.002, 迭代=300, K=2


## 6. NES-VMC 训练循环

In [9]:
# 初始化采样器状态
sampler_rng = jax.random.PRNGKey(21)
sampler_state = nes_sampler.init_state(total_machine, total_params, sampler_rng)

# 训练历史记录
history = {
    'step': [],
    'energy_0st': [],
    'energy_1st': [],
    'energy_2st': [],
    'loss': [],
    'grad_norm': [],
    'log_Psi_mean': [],
}

print('\n' + '='*60)
print('开始 NES-VMC 训练 (K=3)')
print('='*60)
print(f'FCI 基态能量: {E_fcis[0]:.8f} Ha')
print(f'FCI 第一激发态: {E_fcis[1]:.8f} Ha (激发能: {(E_fcis[1]-E_fcis[0])*27.2114:.4f} eV)')
print(f'FCI 第二激发态: {E_fcis[2]:.8f} Ha (激发能: {(E_fcis[2]-E_fcis[0])*27.2114:.4f} eV)')
print('='*60 + '\n')

start_time = time.time()

for step in range(N_ITER):
    # 1. 采样
    samples_raw, sampler_state = nes_sampler.sample(
        machine=total_machine,
        parameters=total_params,
        state=sampler_state,
        chain_length=N_SAMPLES_PER_CHAIN
    )
    
    # 2. 维度重塑
    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, SINGLE_SIZE)
    
    # 3. 计算梯度
    grad, loss_mean, E_L_mean = nes_vmc_gradient(
        ha=ha,
        total_matrix_machine=total_matrix_machine,
        total_machine=total_machine,
        single_machine_list=single_machine_list,
        total_params=total_params,
        x_batch=x_batch
    )
    
    # 4. 计算自然梯度
    grad_flat, grad_unravel_fn = ravel_pytree(grad)
    qgt_reg, _ = compute_qgt(total_machine, total_params, x_batch.reshape(-1, K, n_orbitals*2), diag_shift=0.1)
    
    natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
    natural_grad = grad_unravel_fn(natural_grad_flat)
    grad = natural_grad
    
    # 5. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, total_params)
    total_params = optax.apply_updates(total_params, updates)
    
    # 6. 记录结果
    eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
    grad_norm = jnp.linalg.norm(grad_flat)
    
    log_Psi_batch = total_machine(total_params, x_batch)
    
    history['step'].append(step)
    history['loss'].append(loss_mean)
    history['grad_norm'].append(grad_norm)
    history['energy_0st'].append(eig_vals[0])
    history['energy_1st'].append(eig_vals[1])
    history['energy_2st'].append(eig_vals[2])
    history['log_Psi_mean'].append(log_Psi_batch.mean())
    
    # 7. 打印进度
    if step % 30 == 0 or step == N_ITER - 1:
        print(f"Step {step:4d} | Loss: {loss_mean:.6f} | "
              f"E0={eig_vals[0]:.6f} Ha | E1={eig_vals[1]:.6f} Ha | E2={eig_vals[2]:.6f} Ha | "
              f"grad_norm={grad_norm:.4f}")

end_time = time.time()
print(f'\n训练耗时: {end_time - start_time:.2f} 秒')


开始 NES-VMC 训练 (K=3)
FCI 基态能量: -55.51258121 Ha
FCI 第一激发态: -55.06757883 Ha (激发能: 12.1091 eV)
FCI 第二激发态: -55.00874767 Ha (激发能: 13.7100 eV)

Step    0 | Loss: -70.744483 | E0=-46.422534 Ha | E1=-24.321949 Ha | E2=-24.321949 Ha | grad_norm=55075.3105


KeyboardInterrupt: 

## 7. 结果分析

In [ ]:
# 取最后几个迭代的平均值作为最终结果
n_avg = 20
final_E0 = jnp.mean(jnp.array(history['energy_0st'][-n_avg:]))
final_E1 = jnp.mean(jnp.array(history['energy_1st'][-n_avg:]))
final_E2 = jnp.mean(jnp.array(history['energy_2st'][-n_avg:]))

print('='*60)
print('NES-VMC 计算结果 (K=3)')
print('='*60)
print(f'基态能量 E0 = {final_E0:.8f} Ha (FCI: {E_fcis[0]:.8f} Ha, 误差: {(final_E0-E_fcis[0])*1000:.4f} mHa)')
print(f'第一激发态 E1 = {final_E1:.8f} Ha (FCI: {E_fcis[1]:.8f} Ha, 误差: {(final_E1-E_fcis[1])*1000:.4f} mHa)')
print(f'第二激发态 E2 = {final_E2:.8f} Ha (FCI: {E_fcis[2]:.8f} Ha, 误差: {(final_E2-E_fcis[2])*1000:.4f} mHa)')
print('='*60)
print(f'\n激发能:')
print(f'NES-VMC: E1-E0 = {(final_E1-final_E0)*27.2114:.4f} eV, E2-E0 = {(final_E2-final_E0)*27.2114:.4f} eV')
print(f'FCI:     E1-E0 = {(E_fcis[1]-E_fcis[0])*27.2114:.4f} eV, E2-E0 = {(E_fcis[2]-E_fcis[0])*27.2114:.4f} eV')

In [ ]:
# 绘制训练曲线
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 基态能量
ax = axes[0, 0]
ax.plot(history['energy_0st'], label='NES-VMC E0', color='blue')
ax.axhline(E_fcis[0], color='red', linestyle='--', label=f'FCI E0 = {E_fcis[0]:.4f}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Energy (Ha)')
ax.set_title('Ground State Energy')
ax.legend()
ax.grid(True, alpha=0.3)

# 第一激发态
ax = axes[0, 1]
ax.plot(history['energy_1st'], label='NES-VMC E1', color='blue')
ax.axhline(E_fcis[1], color='red', linestyle='--', label=f'FCI E1 = {E_fcis[1]:.4f}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Energy (Ha)')
ax.set_title('First Excited State Energy')
ax.legend()
ax.grid(True, alpha=0.3)

# 第二激发态
ax = axes[1, 0]
ax.plot(history['energy_2st'], label='NES-VMC E2', color='blue')
ax.axhline(E_fcis[2], color='red', linestyle='--', label=f'FCI E2 = {E_fcis[2]:.4f}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Energy (Ha)')
ax.set_title('Second Excited State Energy')
ax.legend()
ax.grid(True, alpha=0.3)

# Loss
ax = axes[1, 1]
ax.plot(history['loss'], label='Loss', color='green')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.set_title('NES-VMC Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('NH3_NES_VMC_K3_results.png', dpi=150)
plt.show()

print('图像已保存为 NH3_NES_VMC_K3_results.png')

In [ ]:
# 保存训练历史
import pickle
import os

os.makedirs('./data', exist_ok=True)
with open('./data/NH3_NES_VMC_K3_history.pkl', 'wb') as f:
    pickle.dump(history, f)

print('训练历史已保存为 ./data/NH3_NES_VMC_K3_history.pkl')